# Phase 3: Fine-Tuning DistilBERT

## What is DistilBERT?
- Created by Hugging Face as a **distilled** (compressed) version of BERT from Google.
- **Bidirectional** = it reads text both left-to-right AND right-to-left simultaneously, giving it full context for every token. (Unlike GPT which is left-to-right only.)
- 40% smaller than BERT, 60% faster, retains 97% of BERT's performance.

## What is Fine-Tuning?
DistilBERT was pre-trained on Wikipedia + BookCorpus — it already understands language deeply.
Fine-tuning = we add a small classification head on top and train the whole thing on our specific task.
We don't train from scratch — we reuse its language knowledge. This is **transfer learning**.

## Handling Class Imbalance
With ~95% real / ~5% fake, we use **weighted cross-entropy loss**.
The loss function penalizes mistakes on the minority class (fake) more heavily,
forcing the model to actually learn to detect fakes instead of just predicting 'real' always.

Weight formula: `weight_for_class_i = total_samples / (n_classes * samples_in_class_i)`

In [6]:
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
import os

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [7]:
# ── Hyperparameters ────────────────────────────────────────────────────────
# These are the knobs you tune. Keep them here so they're easy to find.
MAX_LEN    = 256   # Token limit — adjust based on 02_Tokenization findings
BATCH_SIZE = 16    # How many samples per gradient update
EPOCHS     = 3     # DistilBERT fine-tuning rarely needs more than 3-5
LR         = 2e-5  # Learning rate — standard for BERT fine-tuning
WARMUP     = 0.1   # First 10% of steps used for LR warm-up

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────
df = pd.read_csv('../data/processed.csv')
df['combined_text'] = df['combined_text'].fillna('').astype(str)

# Stratified split — preserves class ratio in each split
# This is critical with imbalanced data
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['fraudulent'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['fraudulent'], random_state=SEED
)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Train fake %: {train_df['fraudulent'].mean()*100:.1f}%")
print(f"Val   fake %: {val_df['fraudulent'].mean()*100:.1f}%")
print(f"Test  fake %: {test_df['fraudulent'].mean()*100:.1f}%")

Train: 14,304 | Val: 1,788 | Test: 1,788
Train fake %: 4.8%
Val   fake %: 4.9%
Test  fake %: 4.8%


In [9]:
# ── Dataset Class ─────────────────────────────────────────────────────────
# PyTorch requires data to be wrapped in a Dataset class.
# It defines __len__ (how many samples) and __getitem__ (how to get sample i).

tokenizer = DistilBertTokenizer.from_pretrained('/content/models/tokenizer')

class JobPostingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = JobPostingDataset(train_df['combined_text'], train_df['fraudulent'], tokenizer, MAX_LEN)
val_dataset   = JobPostingDataset(val_df['combined_text'],   val_df['fraudulent'],   tokenizer, MAX_LEN)
test_dataset  = JobPostingDataset(test_df['combined_text'],  test_df['fraudulent'],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")

Train batches: 894


In [10]:
# ── Class Weights for Imbalance ───────────────────────────────────────────
# We compute how much MORE we penalize mistakes on the minority class.
n_real = (train_df['fraudulent'] == 0).sum()
n_fake = (train_df['fraudulent'] == 1).sum()
n_total = len(train_df)

weight_real = n_total / (2 * n_real)
weight_fake = n_total / (2 * n_fake)

class_weights = torch.tensor([weight_real, weight_fake], dtype=torch.float).to(device)
print(f"Weight for Real (0): {weight_real:.3f}")
print(f"Weight for Fake (1): {weight_fake:.3f}")
print(f"→ Fake mistakes penalized {weight_fake/weight_real:.1f}x more than Real mistakes")

Weight for Real (0): 0.525
Weight for Fake (1): 10.320
→ Fake mistakes penalized 19.6x more than Real mistakes


In [11]:
# ── Model ─────────────────────────────────────────────────────────────────
# DistilBertForSequenceClassification = DistilBERT + linear classification head
# num_labels=2 → binary classification (real vs fake)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model = model.to(device)

# Loss: weighted cross-entropy handles class imbalance
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer: AdamW is the standard for transformer fine-tuning
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# LR Scheduler: linearly warm up then decay
# Warmup prevents large gradient updates from destabilizing pre-trained weights
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Total training steps : {total_steps}")
print(f"Warmup steps         : {warmup_steps}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total training steps : 2682
Warmup steps         : 268


In [12]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

def evaluate(model, loader, loss_fn, device):
    """Run model on a DataLoader, return loss + metrics."""
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits
            loss    = loss_fn(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss  = total_loss / len(loader)
    # We use macro F1 so both classes contribute equally
    f1        = f1_score(all_labels, all_preds, average='macro')
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    accuracy  = accuracy_score(all_labels, all_preds)

    return avg_loss, accuracy, f1, precision, recall, all_preds, all_labels

In [13]:
# ── Training Loop ─────────────────────────────────────────────────────────
writer    = SummaryWriter('../tensorboard_logs')
best_f1   = 0
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()                          # Clear old gradients
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask) # Forward pass
        loss = loss_fn(outputs.logits, labels)         # Compute loss
        loss.backward()                                # Backpropagation

        # Gradient clipping: prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()   # Update weights
        scheduler.step()   # Update learning rate

        epoch_loss  += loss.item()
        global_step += 1

        # Log training loss to TensorBoard every 50 steps
        if step % 50 == 0:
            writer.add_scalar('Loss/train_step', loss.item(), global_step)
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

    # ── Validation after each epoch ───────────────────────────────────────
    val_loss, val_acc, val_f1, val_prec, val_rec, _, _ = evaluate(
        model, val_loader, loss_fn, device
    )

    train_avg_loss = epoch_loss / len(train_loader)

    # Log epoch metrics to TensorBoard
    writer.add_scalars('Loss/epoch',      {'train': train_avg_loss, 'val': val_loss},  epoch+1)
    writer.add_scalars('Metrics/epoch',   {'accuracy': val_acc, 'f1': val_f1,
                                           'precision': val_prec, 'recall': val_rec},  epoch+1)

    print(f"\nEpoch {epoch+1}/{EPOCHS} Summary:")
    print(f"  Train Loss: {train_avg_loss:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}")
    print(f"  Val   F1  : {val_f1:.4f}")
    print(f"  Val   Acc : {val_acc:.4f}\n")

    # Save best model based on validation F1 (NOT accuracy)
    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained('../models/best_model')
        tokenizer.save_pretrained('../models/best_model')
        print(f"  ✓ New best model saved (F1={best_f1:.4f})")

writer.close()
print('Training complete. Best Val F1:', best_f1)

  Epoch 1 | Step 0/894 | Loss: 0.6666
  Epoch 1 | Step 50/894 | Loss: 0.7508
  Epoch 1 | Step 100/894 | Loss: 0.9779
  Epoch 1 | Step 150/894 | Loss: 0.1043
  Epoch 1 | Step 200/894 | Loss: 1.7513
  Epoch 1 | Step 250/894 | Loss: 0.0094
  Epoch 1 | Step 300/894 | Loss: 0.0127
  Epoch 1 | Step 350/894 | Loss: 0.2839
  Epoch 1 | Step 400/894 | Loss: 1.6625
  Epoch 1 | Step 450/894 | Loss: 1.8779
  Epoch 1 | Step 500/894 | Loss: 0.2213
  Epoch 1 | Step 550/894 | Loss: 0.0114
  Epoch 1 | Step 600/894 | Loss: 0.0311
  Epoch 1 | Step 650/894 | Loss: 2.4972
  Epoch 1 | Step 700/894 | Loss: 1.7096
  Epoch 1 | Step 750/894 | Loss: 0.3191
  Epoch 1 | Step 800/894 | Loss: 1.0480
  Epoch 1 | Step 850/894 | Loss: 0.0031

Epoch 1/3 Summary:
  Train Loss: 0.5016
  Val   Loss: 0.3661
  Val   F1  : 0.9187
  Val   Acc : 0.9860



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best model saved (F1=0.9187)
  Epoch 2 | Step 0/894 | Loss: 2.9037
  Epoch 2 | Step 50/894 | Loss: 0.0063
  Epoch 2 | Step 100/894 | Loss: 0.0029
  Epoch 2 | Step 150/894 | Loss: 0.0015
  Epoch 2 | Step 200/894 | Loss: 0.1271
  Epoch 2 | Step 250/894 | Loss: 0.0052
  Epoch 2 | Step 300/894 | Loss: 0.0020
  Epoch 2 | Step 350/894 | Loss: 0.8313
  Epoch 2 | Step 400/894 | Loss: 0.0038
  Epoch 2 | Step 450/894 | Loss: 0.0017
  Epoch 2 | Step 500/894 | Loss: 0.0014
  Epoch 2 | Step 550/894 | Loss: 0.0022
  Epoch 2 | Step 600/894 | Loss: 0.0105
  Epoch 2 | Step 650/894 | Loss: 0.0012
  Epoch 2 | Step 700/894 | Loss: 0.0034
  Epoch 2 | Step 750/894 | Loss: 0.0159
  Epoch 2 | Step 800/894 | Loss: 0.0103
  Epoch 2 | Step 850/894 | Loss: 0.0028

Epoch 2/3 Summary:
  Train Loss: 0.2254
  Val   Loss: 0.3124
  Val   F1  : 0.9286
  Val   Acc : 0.9871



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best model saved (F1=0.9286)
  Epoch 3 | Step 0/894 | Loss: 0.0011
  Epoch 3 | Step 50/894 | Loss: 0.0014
  Epoch 3 | Step 100/894 | Loss: 0.0009
  Epoch 3 | Step 150/894 | Loss: 0.0019
  Epoch 3 | Step 200/894 | Loss: 0.0010
  Epoch 3 | Step 250/894 | Loss: 0.0008
  Epoch 3 | Step 300/894 | Loss: 0.0011
  Epoch 3 | Step 350/894 | Loss: 0.0009
  Epoch 3 | Step 400/894 | Loss: 0.0027
  Epoch 3 | Step 450/894 | Loss: 0.0011
  Epoch 3 | Step 500/894 | Loss: 0.0026
  Epoch 3 | Step 550/894 | Loss: 0.0012
  Epoch 3 | Step 600/894 | Loss: 1.1012
  Epoch 3 | Step 650/894 | Loss: 0.0005
  Epoch 3 | Step 700/894 | Loss: 0.0006
  Epoch 3 | Step 750/894 | Loss: 0.0006
  Epoch 3 | Step 800/894 | Loss: 0.0009
  Epoch 3 | Step 850/894 | Loss: 0.0004

Epoch 3/3 Summary:
  Train Loss: 0.0979
  Val   Loss: 0.3472
  Val   F1  : 0.9313
  Val   Acc : 0.9877



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best model saved (F1=0.9313)
Training complete. Best Val F1: 0.931296109993293
